In [3]:
import os

os.chdir(r"E:\PGT\MLP\project_materials")
print(os.getcwd())

E:\PGT\MLP\project_materials


In [4]:
# %%
# # Add any additional libraries or submodules below

# Data libraries
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules
import sklearn
# Load data  
df = pd.read_csv("unicef_malawi.csv")
df.head()



,HH1,HH2,LN,FS4,CB3,CB4,CB5A,CB5B,CB7,CB11,...,HC19,TN1,WS1,WS3,WS4,WS7,WS11,WS14,WS15,HW5
0,1.0,2.0,2.0,2.0,14.0,YES,PRIMARY,CLASS/GRADE 6,YES,NO,...,NO,NO,PIPED WATER: PUBLIC TAP / STANDPIPE,ELSEWHERE,5.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,ELSEWHERE,YES,NO
1,1.0,3.0,1.0,1.0,5.0,YES,ECE,NaN,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,30.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITHOUT SLAB / OPEN PIT,IN OWN YARD / PLOT,NO,YES
2,1.0,4.0,2.0,2.0,16.0,YES,PRIMARY,CLASS/GRADE 7,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,6.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,YES
3,1.0,8.0,2.0,2.0,13.0,YES,PRIMARY,CLASS/GRADE 7,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,30.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,NO
4,1.0,10.0,2.0,2.0,14.0,YES,PRIMARY,CLASS/GRADE 6,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,8.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,NO


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import chi2_contingency

# =========================================================
# 0. 基础设置
# =========================================================

input_file = "unicef_malawi_cleaned_full.csv"
target_col = "FCF26"

test_size = 0.2
random_state = 42

# 初筛阈值
mi_thresh = 0.001
chi2_p_thresh = 0.05

# 输出文件
output_screening_table = "screening_results_train_only.csv"
output_selected_vars = "prelim_selected_variables.csv"
output_train_screened = "train_screened.csv"
output_test_screened = "test_screened.csv"

# =========================================================
# 1. 读入数据 + 目标变量处理
# =========================================================

df = pd.read_csv(input_file)

# 只保留目标不缺失的样本
df = df.dropna(subset=[target_col]).copy()

# 目标变量二分类
y_map = {"No": 0, "YES": 1}
df["y"] = df[target_col].map(y_map)

df = df.dropna(subset=["y"]).copy()
df["y"] = df["y"].astype(int)

print("数据 shape:", df.shape)
print(df["y"].value_counts(dropna=False))

# =========================================================
# 2. 变量分组
#    这里沿用你之前的分组，可再按实际调整
# =========================================================

numeric_vars = [
    "CB3", "CB5B", "CL3", "CL13",
    "wscore", "WB4", "WB6B", "MA2",
    "CSURV", "CDEAD"
]

ordinal_vars = [
    "CB5A", "WB6A", "WB14",
    "AF10", "AF11", "AF12",
    "LS1", "LS2", "LS3", "LS4",
    "TA14", "TN1"
]

categorical_vars = [
    "CB4", "CB7", "CB11", "HH6", "HH7", "HL4", "ethnicity",
    "CL2", "CL12", "CL3_status", "CL13_status",
    "FCD2A", "FCD2B", "FCD2C", "FCD2D", "FCD2E", "FCD2F",
    "FCD2G", "FCD2H", "FCD2I", "FCD2J", "FCD2K", "FCD5",
    "DV1A", "DV1B", "DV1C", "DV1D", "DV1E",
    "VT1", "VT9", "VT20", "VT21", "VT22A", "VT22B", "VT22C",
    "VT22D", "VT22E", "VT22F", "VT22X",
    "MSTATUS", "MA2_status", "MA3",
    "disability", "TA1",
    "HC4", "HC5", "HC8", "HC11", "HC12", "HC13", "HC14",
    "HC15", "HC17", "HC19",
    "WS1", "WS11", "HW5"
]

def keep_existing_unique(cols, df_columns):
    seen = set()
    out = []
    for c in cols:
        if c in df_columns and c not in seen:
            out.append(c)
            seen.add(c)
    return out

numeric_vars = keep_existing_unique(numeric_vars, df.columns)
ordinal_vars = keep_existing_unique(ordinal_vars, df.columns)
categorical_vars = [c for c in categorical_vars if c in df.columns and c not in set(numeric_vars + ordinal_vars)]
categorical_vars = keep_existing_unique(categorical_vars, df.columns)

all_feature_vars = numeric_vars + ordinal_vars + categorical_vars

# =========================================================
# 3. 先划分 train / test，避免泄露
# =========================================================

X = df[all_feature_vars].copy()
y = df["y"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

print("\n训练集 shape:", X_train.shape)
print("测试集 shape:", X_test.shape)

# =========================================================
# 4. 只在训练集上拟合插补器，然后应用到 train/test
# =========================================================

X_train_imp = X_train.copy()
X_test_imp = X_test.copy()

# ---------- 数值变量：中位数插补 ----------
for col in numeric_vars:
    X_train_imp[col] = pd.to_numeric(X_train_imp[col], errors="coerce")
    X_test_imp[col] = pd.to_numeric(X_test_imp[col], errors="coerce")

if len(numeric_vars) > 0:
    num_imputer = SimpleImputer(strategy="median")
    X_train_imp[numeric_vars] = pd.DataFrame(
        num_imputer.fit_transform(X_train_imp[numeric_vars]),
        columns=numeric_vars,
        index=X_train_imp.index
    )
    X_test_imp[numeric_vars] = pd.DataFrame(
        num_imputer.transform(X_test_imp[numeric_vars]),
        columns=numeric_vars,
        index=X_test_imp.index
    )

# ---------- 有序变量：先众数插补 ----------
if len(ordinal_vars) > 0:
    ord_imputer = SimpleImputer(strategy="most_frequent")
    X_train_imp[ordinal_vars] = pd.DataFrame(
        ord_imputer.fit_transform(X_train_imp[ordinal_vars]),
        columns=ordinal_vars,
        index=X_train_imp.index
    )
    X_test_imp[ordinal_vars] = pd.DataFrame(
        ord_imputer.transform(X_test_imp[ordinal_vars]),
        columns=ordinal_vars,
        index=X_test_imp.index
    )

# ---------- 分类变量：众数插补 ----------
if len(categorical_vars) > 0:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    X_train_imp[categorical_vars] = pd.DataFrame(
        cat_imputer.fit_transform(X_train_imp[categorical_vars]),
        columns=categorical_vars,
        index=X_train_imp.index
    )
    X_test_imp[categorical_vars] = pd.DataFrame(
        cat_imputer.transform(X_test_imp[categorical_vars]),
        columns=categorical_vars,
        index=X_test_imp.index
    )

# =========================================================
# 5. 有序变量映射（只用于 MI）
# =========================================================

ordinal_mappings = {
    "CB5A": {
        "NoSchool": 0,
        "PRE-PRIMARY": 1,
        "PRIMARY": 2,
        "LOWER SECONDARY": 3,
        "UPPER SECONDARY": 4,
        "HIGHER": 5
    },
    "WB6A": {
        "NoSchool": 0,
        "PRE-PRIMARY": 1,
        "PRIMARY": 2,
        "LOWER SECONDARY": 3,
        "UPPER SECONDARY": 4,
        "HIGHER": 5
    },
    "WB14": {
        "CANNOT READ AT ALL": 0,
        "ABLE TO READ ONLY PARTS OF SENTENCE": 1,
        "ABLE TO READ WHOLE SENTENCE": 2
    },
    "AF10": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2,
        "CANNOT DO AT ALL": 3
    },
    "AF11": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2,
        "CANNOT DO AT ALL": 3
    },
    "AF12": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2,
        "CANNOT DO AT ALL": 3
    },
    "LS1": {
        "VERY HAPPY": 4,
        "SOMEWHAT HAPPY": 3,
        "NEITHER HAPPY NOR UNHAPPY": 2,
        "SOMEWHAT UNHAPPY": 1,
        "VERY UNHAPPY": 0
    },
    "LS2": {
        "VERY SATISFIED": 4,
        "SOMEWHAT SATISFIED": 3,
        "NEITHER SATISFIED NOR UNSATISFIED": 2,
        "SOMEWHAT UNSATISFIED": 1,
        "VERY UNSATISFIED": 0
    },
    "LS3": {
        "MUCH BETTER": 4,
        "SOMEWHAT BETTER": 3,
        "ABOUT THE SAME": 2,
        "SOMEWHAT WORSE": 1,
        "MUCH WORSE": 0
    },
    "LS4": {
        "MUCH BETTER": 4,
        "SOMEWHAT BETTER": 3,
        "ABOUT THE SAME": 2,
        "SOMEWHAT WORSE": 1,
        "MUCH WORSE": 0
    },
    "TA14": {
        "NEVER": 0,
        "LESS THAN ONCE A WEEK": 1,
        "AT LEAST ONCE A WEEK": 2,
        "DAILY": 3
    },
    "TN1": {
        "NO": 0,
        "YES": 1
    }
}

def encode_ordinal(df_in, ordinal_vars, ordinal_mappings):
    out = pd.DataFrame(index=df_in.index)
    valid_cols = []
    for col in ordinal_vars:
        if col in ordinal_mappings:
            out[col] = df_in[col].map(ordinal_mappings[col])
        else:
            out[col] = pd.Categorical(df_in[col]).codes
            out.loc[out[col] < 0, col] = np.nan

        if not out[col].isna().all():
            valid_cols.append(col)
    return out, valid_cols

X_train_ord, valid_ordinal_vars = encode_ordinal(X_train_imp, ordinal_vars, ordinal_mappings)
X_test_ord, _ = encode_ordinal(X_test_imp, ordinal_vars, ordinal_mappings)

if len(valid_ordinal_vars) > 0:
    ord2_imputer = SimpleImputer(strategy="most_frequent")
    X_train_ord[valid_ordinal_vars] = pd.DataFrame(
        ord2_imputer.fit_transform(X_train_ord[valid_ordinal_vars]),
        columns=valid_ordinal_vars,
        index=X_train_ord.index
    )
    X_test_ord[valid_ordinal_vars] = pd.DataFrame(
        ord2_imputer.transform(X_test_ord[valid_ordinal_vars]),
        columns=valid_ordinal_vars,
        index=X_test_ord.index
    )

# =========================================================
# 6. 训练集上做 MI（数值 + 有序）
# =========================================================

mi_feature_df = pd.DataFrame(index=X_train_imp.index)

for col in numeric_vars:
    mi_feature_df[col] = X_train_imp[col]

for col in valid_ordinal_vars:
    mi_feature_df[col] = X_train_ord[col]

if mi_feature_df.shape[1] > 0:
    discrete_mask = [col in valid_ordinal_vars for col in mi_feature_df.columns]

    mi_scores = mutual_info_classif(
        mi_feature_df,
        y_train,
        discrete_features=discrete_mask,
        random_state=random_state
    )

    mi_table = pd.DataFrame({
        "variable": mi_feature_df.columns,
        "test_type": "MI",
        "score": mi_scores,
        "p_value": np.nan,
        "selected": mi_scores >= mi_thresh
    }).sort_values(by="score", ascending=False)
else:
    mi_table = pd.DataFrame(columns=["variable", "test_type", "score", "p_value", "selected"])

# =========================================================
# 7. 训练集上做卡方（分类变量）
# =========================================================

chi2_rows = []

for col in categorical_vars:
    tmp = pd.DataFrame({
        col: X_train_imp[col],
        "y": y_train
    })

    if tmp[col].nunique(dropna=False) <= 1:
        chi2_rows.append({
            "variable": col,
            "test_type": "Chi-square",
            "score": np.nan,
            "p_value": np.nan,
            "selected": False
        })
        continue

    contingency = pd.crosstab(tmp[col], tmp["y"])

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        chi2_rows.append({
            "variable": col,
            "test_type": "Chi-square",
            "score": np.nan,
            "p_value": np.nan,
            "selected": False
        })
        continue

    chi2, p, dof, expected = chi2_contingency(contingency)

    chi2_rows.append({
        "variable": col,
        "test_type": "Chi-square",
        "score": chi2,
        "p_value": p,
        "selected": p < chi2_p_thresh
    })

chi2_table = pd.DataFrame(chi2_rows).sort_values(by="p_value", ascending=True)

# =========================================================
# 8. 合并初筛结果
# =========================================================

screening_table = pd.concat([mi_table, chi2_table], axis=0, ignore_index=True)
prelim_selected_vars = screening_table.loc[screening_table["selected"], "variable"].tolist()

print("\n初步筛选后变量数:", len(prelim_selected_vars))
print(prelim_selected_vars)

# 训练集 / 测试集都只保留这些变量
X_train_screened = X_train[prelim_selected_vars].copy()
X_test_screened = X_test[prelim_selected_vars].copy()

train_screened_df = X_train_screened.copy()
train_screened_df["y"] = y_train
test_screened_df = X_test_screened.copy()
test_screened_df["y"] = y_test

screening_table.to_csv(output_screening_table, index=False, encoding="utf-8-sig")
pd.DataFrame({"selected_variable": prelim_selected_vars}).to_csv(output_selected_vars, index=False, encoding="utf-8-sig")
train_screened_df.to_csv(output_train_screened, index=False, encoding="utf-8-sig")
test_screened_df.to_csv(output_test_screened, index=False, encoding="utf-8-sig")

print("\n已输出：")
print(output_screening_table)
print(output_selected_vars)
print(output_train_screened)
print(output_test_screened)

数据 shape: (13036, 91)
y
1    7131
0    5905
Name: count, dtype: int64

训练集 shape: (10428, 79)
测试集 shape: (2608, 79)

初步筛选后变量数: 50
['CL13', 'MA2', 'CB5B', 'WB4', 'LS1', 'AF10', 'WB6B', 'WB6A', 'VT22A', 'VT22B', 'VT20', 'VT21', 'ethnicity', 'HH7', 'CL2', 'CL3_status', 'FCD2H', 'VT1', 'FCD2E', 'MA3', 'CL12', 'CL13_status', 'MSTATUS', 'VT22X', 'MA2_status', 'FCD2D', 'FCD2A', 'HC12', 'VT22C', 'disability', 'FCD2B', 'HC11', 'WS1', 'HC19', 'VT9', 'WS11', 'FCD2J', 'HC8', 'VT22F', 'HW5', 'HC5', 'HC4', 'VT22E', 'FCD2I', 'FCD2G', 'DV1B', 'VT22D', 'HC14', 'HC13', 'DV1C']

已输出：
screening_results_train_only.csv
prelim_selected_variables.csv
train_screened.csv
test_screened.csv


In [6]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

# =========================================================
# 0. 读入第一块输出的数据
# =========================================================

train_df = pd.read_csv("train_screened.csv")
test_df = pd.read_csv("test_screened.csv")

target_col = "y"
random_state = 42

X_train = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].copy()

X_test = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].copy()

baseline_vars = X_train.columns.tolist()

print("进入 baseline 的变量数:", len(baseline_vars))
print(baseline_vars)

# =========================================================
# 1. 识别变量类型
# =========================================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]

print("\n数值变量数:", len(numeric_cols))
print("分类变量数:", len(categorical_cols))

# =========================================================
# 2. 预处理 pipeline
# =========================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# =========================================================
# 3. Baseline：普通逻辑回归
#    如果你的 sklearn 版本支持 penalty=None，就是真正无正则
# =========================================================

baseline_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
    random_state=random_state
)

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", baseline_model)
])

baseline_pipeline.fit(X_train, y_train)

# =========================================================
# 4. 评估函数
# =========================================================

def evaluate_model(model, X, y, dataset_name):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    result = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }
    return result, y_pred, y_prob

baseline_train_metrics, baseline_train_pred, baseline_train_prob = evaluate_model(
    baseline_pipeline, X_train, y_train, "train"
)
baseline_test_metrics, baseline_test_pred, baseline_test_prob = evaluate_model(
    baseline_pipeline, X_test, y_test, "test"
)

baseline_perf_df = pd.DataFrame([baseline_train_metrics, baseline_test_metrics])

print("\nBaseline表现：")
print(baseline_perf_df)

# =========================================================
# 5. 提取 baseline 系数
# =========================================================

feature_names_after_encoding = baseline_pipeline.named_steps["preprocessor"].get_feature_names_out()
coefs = baseline_pipeline.named_steps["model"].coef_[0]

baseline_coef_df = pd.DataFrame({
    "feature_after_encoding": feature_names_after_encoding,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs)
}).sort_values("abs_coefficient", ascending=False)

# =========================================================
# 6. 导出 baseline 预测结果
# =========================================================

baseline_train_pred_df = pd.DataFrame({
    "y_true": y_train,
    "y_pred": baseline_train_pred,
    "y_prob_1": baseline_train_prob
}, index=X_train.index)

baseline_test_pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": baseline_test_pred,
    "y_prob_1": baseline_test_prob
}, index=X_test.index)

pd.DataFrame({"baseline_variable": baseline_vars}).to_csv(
    "baseline_variables.csv", index=False, encoding="utf-8-sig"
)
baseline_coef_df.to_csv(
    "baseline_coefficients.csv", index=False, encoding="utf-8-sig"
)
baseline_perf_df.to_csv(
    "baseline_performance.csv", index=False, encoding="utf-8-sig"
)
baseline_train_pred_df.to_csv(
    "baseline_train_predictions.csv", index=False, encoding="utf-8-sig"
)
baseline_test_pred_df.to_csv(
    "baseline_test_predictions.csv", index=False, encoding="utf-8-sig"
)

print("\n已输出：")
print("baseline_variables.csv")
print("baseline_coefficients.csv")
print("baseline_performance.csv")
print("baseline_train_predictions.csv")
print("baseline_test_predictions.csv")

进入 baseline 的变量数: 50
['CL13', 'MA2', 'CB5B', 'WB4', 'LS1', 'AF10', 'WB6B', 'WB6A', 'VT22A', 'VT22B', 'VT20', 'VT21', 'ethnicity', 'HH7', 'CL2', 'CL3_status', 'FCD2H', 'VT1', 'FCD2E', 'MA3', 'CL12', 'CL13_status', 'MSTATUS', 'VT22X', 'MA2_status', 'FCD2D', 'FCD2A', 'HC12', 'VT22C', 'disability', 'FCD2B', 'HC11', 'WS1', 'HC19', 'VT9', 'WS11', 'FCD2J', 'HC8', 'VT22F', 'HW5', 'HC5', 'HC4', 'VT22E', 'FCD2I', 'FCD2G', 'DV1B', 'VT22D', 'HC14', 'HC13', 'DV1C']

数值变量数: 2
分类变量数: 48

Baseline表现：
  dataset  accuracy  balanced_accuracy  precision    recall        f1  \
0   train  0.625719           0.614956   0.638092  0.729488  0.680736   
1    test  0.609663           0.597327   0.622528  0.728101  0.671189   

    roc_auc  
0  0.661797  
1  0.640173  

已输出：
baseline_variables.csv
baseline_coefficients.csv
baseline_performance.csv
baseline_train_predictions.csv
baseline_test_predictions.csv


In [7]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

# =========================================================
# 0. 读入筛选后的 train / test
# =========================================================

train_df = pd.read_csv("train_screened.csv")
test_df = pd.read_csv("test_screened.csv")

target_col = "y"
random_state = 42
Cs_grid = np.logspace(-2, 2, 30)
X_train = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].copy()

X_test = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].copy()

# =========================================================
# 1. 识别变量类型
# =========================================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]

# =========================================================
# 2. 预处理 pipeline
# =========================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# =========================================================
# 3. Final：L1 逻辑回归，训练集内部CV选C
# =========================================================

final_model = LogisticRegressionCV(
    Cs=Cs_grid,
    cv=5,
    penalty="l1",
    solver="saga",
    scoring="roc_auc",
    max_iter=8000,
    random_state=random_state,
    n_jobs=-1,
    refit=True
)

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", final_model)
])

final_pipeline.fit(X_train, y_train)

best_C_final = final_pipeline.named_steps["model"].C_[0]
print("\nFinal(L1)最优 C:", best_C_final)

# =========================================================
# 4. 评估函数
# =========================================================

def evaluate_model(model, X, y, dataset_name):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    result = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }
    return result, y_pred, y_prob

final_train_metrics, final_train_pred, final_train_prob = evaluate_model(
    final_pipeline, X_train, y_train, "train"
)
final_test_metrics, final_test_pred, final_test_prob = evaluate_model(
    final_pipeline, X_test, y_test, "test"
)

final_perf_df = pd.DataFrame([final_train_metrics, final_test_metrics])
print("\nFinal(L1)表现：")
print(final_perf_df)

# =========================================================
# 5. 提取 L1 非零变量
# =========================================================

feature_names_after_encoding = final_pipeline.named_steps["preprocessor"].get_feature_names_out()
coefs = final_pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature_after_encoding": feature_names_after_encoding,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs)
})

selected_coef_df = coef_df[coef_df["coefficient"] != 0].copy()
selected_coef_df = selected_coef_df.sort_values("abs_coefficient", ascending=False)

# 还原到原始变量层面
original_feature_names = X_train.columns.tolist()

def recover_original_variable(encoded_name, original_cols):
    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "")
    elif encoded_name.startswith("cat__"):
        temp = encoded_name.replace("cat__", "")
        sorted_cols = sorted(original_cols, key=len, reverse=True)
        for col in sorted_cols:
            if temp == col or temp.startswith(col + "_"):
                return col
        return temp
    else:
        return encoded_name

selected_coef_df["original_variable"] = selected_coef_df["feature_after_encoding"].apply(
    lambda x: recover_original_variable(x, original_feature_names)
)

final_original_vars = selected_coef_df["original_variable"].drop_duplicates().tolist()

print("\n进入 final 的原始变量数:", len(final_original_vars))
print(final_original_vars)

# =========================================================
# 6. 导出 final 预测结果
# =========================================================

final_train_pred_df = pd.DataFrame({
    "y_true": y_train,
    "y_pred": final_train_pred,
    "y_prob_1": final_train_prob
}, index=X_train.index)

final_test_pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": final_test_pred,
    "y_prob_1": final_test_prob
}, index=X_test.index)

pd.DataFrame({"final_variable": final_original_vars}).to_csv(
    "final_l1_variables.csv", index=False, encoding="utf-8-sig"
)
selected_coef_df.to_csv("final_l1_nonzero_coefficients.csv", index=False, encoding="utf-8-sig")
final_perf_df.to_csv("final_l1_performance.csv", index=False, encoding="utf-8-sig")
final_train_pred_df.to_csv("final_l1_train_predictions.csv", index=False, encoding="utf-8-sig")
final_test_pred_df.to_csv("final_l1_test_predictions.csv", index=False, encoding="utf-8-sig")

print("\n已输出：")
print("final_l1_variables.csv")
print("final_l1_nonzero_coefficients.csv")
print("final_l1_performance.csv")
print("final_l1_train_predictions.csv")
print("final_l1_test_predictions.csv")



Final(L1)最优 C: 0.06723357536499334

Final(L1)表现：
  dataset  accuracy  balanced_accuracy  precision    recall        f1  \
0   train  0.614404           0.601012   0.623768  0.743513  0.678397   
1    test  0.608129           0.595050   0.619893  0.733707  0.672015   

    roc_auc  
0  0.650714  
1  0.639423  

进入 final 的原始变量数: 38
['VT22A', 'VT22B', 'ethnicity', 'AF10', 'LS1', 'VT20', 'VT1', 'WS1', 'HH7', 'VT21', 'FCD2H', 'MA2_status', 'WS11', 'FCD2E', 'HW5', 'CL3_status', 'CL2', 'HC12', 'HC4', 'MSTATUS', 'MA2', 'FCD2D', 'FCD2A', 'CL12', 'CL13_status', 'CB5B', 'CL13', 'MA3', 'VT22X', 'FCD2B', 'HC19', 'DV1B', 'WB4', 'HC8', 'HC11', 'WB6B', 'FCD2J', 'WB6A']

已输出：
final_l1_variables.csv
final_l1_nonzero_coefficients.csv
final_l1_performance.csv
final_l1_train_predictions.csv
final_l1_test_predictions.csv
